# Supplementary Movie 3: the scar of Fig. 3 in a reduced Hilbert space

The scar column of Fig. 3's network, with its Hilbert-space diagram
replaced by the reduced Hilbert space of the three spins with the largest
`<Z_i(t)>` oscillation. Node colour is the marginal probability of each of
the 2**3 configurations (the other nine spins traced out), laid out by
Hamming distance from the initial configuration. 60 s at 30 fps; needs
ffmpeg.

In [ ]:
%matplotlib inline

import colorcet as cc
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FFMpegWriter, FuncAnimation
from matplotlib.gridspec import GridSpec

from qising_networks import FIGURES_DIR as FIGURES
from qising_networks import PROJECT_ROOT
from qising_networks import scar_metrics as sm
from qising_networks.plots.common import CLASS_COLORS, CLASS_LABELS
from qising_networks.plots.dynamics import (
    draw_network,
    load_examples,
    network_layout,
    network_ylim,
    probabilities_at,
    sector_hamiltonian,
)
from qising_networks.scar_metrics import DISPLAY_TIME_MAX

mpl.rc_file(PROJECT_ROOT / "matplotlibrc")

In [ ]:
# The random-network example of Fig. 3; only its scar column is used.
d = load_examples("ferro")
net_im, net_h, net_sec_idx = d["net_im"], d["net_h"], d["net_sec_idx"]
N = d["N"]
print("network:", d["net_path"].name, "N=", N)

scar_col = {
    "cls": int(sm.SCAR),
    "alpha": int(d["net_alphas"][sm.SCAR]),
    "sec_idx": net_sec_idx,
    "A": d["net_A"],
    "states": d["net_states"],
    "z": d["net_z"][sm.SCAR],
    "fid": d["net_fid"][sm.SCAR],
}
print(
    "Scar column: alpha =",
    scar_col["alpha"],
    " sector dim =",
    len(net_sec_idx),
)

In [ ]:
# The three spins with the largest peak-to-peak <Z_i(t)>.
z_scar = scar_col["z"]
osc_amplitude = z_scar.max(axis=1) - z_scar.min(axis=1)
N_ACTIVE = 3
N_REDUCED = 2**N_ACTIVE
active_spins = np.sort(np.argsort(-osc_amplitude)[:N_ACTIVE])
print(f"{N_ACTIVE} most-oscillating spins (1-based):", active_spins + 1)
print(
    "their peak-to-peak <Z_i(t)> amplitude:",
    np.round(osc_amplitude[active_spins], 3),
)
ranked = np.sort(osc_amplitude)[::-1]
print("amplitude spectrum, all 12 spins, descending:", np.round(ranked, 3))


def reduced_codes(states, ref_idx, spin_positions):
    """Per-basis-state reduced configuration code (0..2**k - 1): bit b
    set iff spin `spin_positions[b]` differs from the reference
    `ref_idx`. The reference always codes to 0 by construction.
    """
    ref_vec = states[ref_idx, spin_positions]
    rel = (states[:, spin_positions] != ref_vec).astype(int)
    return rel @ (1 << np.arange(len(spin_positions)))


def hamming_projection_layout_reduced(n_active):
    """Hamming-distance layout of the 2**n_active reduced configurations:
    x = number of active spins flipped from the reference, y = spread
    within each shell; edges are single flips."""
    n_conf = 2**n_active
    codes_r = np.arange(n_conf)
    weight = np.array([c.bit_count() for c in codes_r], dtype=float)
    XY = np.zeros((n_conf, 2))
    XY[:, 0] = weight
    for w in np.unique(weight):
        members = np.flatnonzero(weight == w)
        m = len(members)
        XY[members, 1] = np.arange(m) - (m - 1) / 2.0
    EDGES = np.array(
        [
            (u, u ^ (1 << bit))
            for u in range(n_conf)
            for bit in range(n_active)
            if u < (u ^ (1 << bit))
        ]
    )
    return XY, EDGES

In [ ]:
FPS = 30
DURATION_S = 60
N_FRAMES = FPS * DURATION_S
frame_times = np.linspace(0.0, DISPLAY_TIME_MAX, N_FRAMES)

lay_net = network_layout(net_im, net_h, use_rcm=True)
SHARED_YLIM = network_ylim(lay_net, h_len=0.90, arrow_gap=0.45)


H_net_m = sector_hamiltonian(net_im, net_h, net_sec_idx)
evals_net_m, evecs_net_m = np.linalg.eigh(H_net_m)
prob = probabilities_at(
    evals_net_m, evecs_net_m, scar_col["alpha"], frame_times
)
fid_t = prob[scar_col["alpha"], :]
z_t = scar_col["states"].T @ prob

# Check against the stored fidelity.
static_time = np.linspace(0.0, DISPLAY_TIME_MAX, scar_col["fid"].shape[0])
prob_chk = probabilities_at(
    evals_net_m, evecs_net_m, scar_col["alpha"], static_time
)
err = np.max(np.abs(prob_chk[scar_col["alpha"]] - scar_col["fid"]))
assert err < 1e-6, f"recompute mismatch: {err}"
print(f"recompute verified against stored fidelity_of_t (max err {err:.2e})")

# Marginal probability of each reduced configuration: AGG sums
# |<s|psi(t)>|^2 over the full states s that share it.
codes = reduced_codes(scar_col["states"], scar_col["alpha"], active_spins)
XY_red, EDGES_red = hamming_projection_layout_reduced(N_ACTIVE)
AGG = np.zeros((N_REDUCED, len(scar_col["sec_idx"])))
AGG[codes, np.arange(len(scar_col["sec_idx"]))] = 1.0
marg_prob = AGG @ prob  # (N_REDUCED, N_FRAMES); each column sums to 1
alpha_code = int(codes[scar_col["alpha"]])
assert alpha_code == 0  # reference is coded 0 by construction

In [ ]:
# Order in which the active spins flip (quoted in the Movie 3 caption).
s0 = scar_col["states"][scar_col["alpha"]]
flip_prob = 0.5 * (1 - z_t * s0[:, None])
for i in active_spins:
    above = np.flatnonzero(flip_prob[i] > 0.5)
    first = f"Jt = {frame_times[above[0]]:.1f}" if len(above) else "never"
    print(
        f"spin {i + 1}: flip probability first > 0.5 at {first}, "
        f"max {flip_prob[i].max():.2f}"
    )
print("correlation of flip probabilities (spins", active_spins + 1, "):")
print(np.round(np.corrcoef(flip_prob[active_spins]), 2))

In [ ]:
colour = CLASS_COLORS[scar_col["cls"]]

cmap_z = cc.cm.CET_D1A
norm_z = mcolors.Normalize(vmin=-1, vmax=1)
# As render_movie.py: CET_L18 with white for unoccupied nodes.
cmap_occ_m = cc.cm.CET_L18.copy()
cmap_occ_m.set_under("white")
cmap_occ_m.set_over(cmap_occ_m(1.0))
# Linear scale up to about the 99th percentile of the marginals.
norm_occ_m = mcolors.Normalize(vmin=0.0, vmax=0.4, clip=False)

HEIGHT_RATIOS = [0.80, 0.40, 0.70, 0.90]
FIG_W = 3.0  # inches
FIG_H = 6.4

fig_m = plt.figure(figsize=(FIG_W, FIG_H))
gs_m = GridSpec(4, 1, figure=fig_m, height_ratios=HEIGHT_RATIOS)
fig_m.set_constrained_layout_pads(
    h_pad=0.02, w_pad=0.02, hspace=0.08, wspace=0.0
)
fig_m.set_layout_engine("constrained")

axn_m = fig_m.add_subplot(gs_m[0, 0])
axf_m = fig_m.add_subplot(gs_m[1, 0])
axs_m = fig_m.add_subplot(gs_m[2, 0], sharex=axf_m)
axh_m = fig_m.add_subplot(gs_m[3, 0])

draw_network(
    axn_m,
    lay_net,
    scar_col["states"][scar_col["alpha"]],
    ylim=SHARED_YLIM,
    arrow_scale=2.5,
)
axn_m.set_title(
    f"{CLASS_LABELS[scar_col['cls']]}: spins "
    f"{{{', '.join(str(i + 1) for i in active_spins)}}} only",
    color=colour,
    pad=2,
)

axf_m.set_xlim(frame_times[0], frame_times[-1])
axf_m.set_ylim(-0.05, 1.10)
axf_m.set_yticks([0, 1])
axf_m.tick_params(labelbottom=False)
axf_m.set_ylabel(r"Fidelity $F(t)$")
(line_m,) = axf_m.plot([], [], lw=0.7, color=colour)

n_spin = z_t.shape[0]
z_masked0 = np.ma.masked_array(z_t, mask=True)
mesh_m = axs_m.pcolormesh(
    frame_times,
    np.arange(1, n_spin + 1),
    z_masked0,
    cmap=cmap_z,
    norm=norm_z,
    shading="nearest",
    rasterized=True,
)
axs_m.set_xlim(frame_times[0], frame_times[-1])
axs_m.set_xlabel(r"Time $Jt$")
axs_m.set_yticks(np.arange(1, n_spin + 1))
axs_m.set_ylabel(r"Spin $i$")
# Highlight the active spins' tick labels.
for i, tick in enumerate(axs_m.get_yticklabels()):
    if i in active_spins:
        tick.set_color(colour)
        tick.set_fontweight("bold")

axh_m.axis("off")
axh_m.add_collection(
    plt.matplotlib.collections.LineCollection(
        XY_red[EDGES_red], colors="0.85", linewidths=0.4, zorder=1
    )
)
sc_m = axh_m.scatter(
    XY_red[:, 0],
    XY_red[:, 1],
    s=10.0,
    c=np.maximum(marg_prob[:, 0], 1e-6),
    cmap=cmap_occ_m,
    norm=norm_occ_m,
    linewidth=0,
    zorder=3,
)
axh_m.scatter(
    XY_red[alpha_code, 0],
    XY_red[alpha_code, 1],
    s=64,
    facecolor="none",
    edgecolor=colour,
    linewidth=1.2,
    zorder=5,
)
# No equal aspect: x and y are different quantities.
x_span = XY_red[:, 0].max() - XY_red[:, 0].min()
y_span = max(XY_red[:, 1].max() - XY_red[:, 1].min(), 1e-9)
axh_m.set_xlim(
    XY_red[:, 0].min() - 0.06 * x_span, XY_red[:, 0].max() + 0.06 * x_span
)
axh_m.set_ylim(
    XY_red[:, 1].min() - 0.10 * y_span, XY_red[:, 1].max() + 0.10 * y_span
)
axh_m.margins(0)
hilbert_axes_info = [(axh_m, XY_red, alpha_code)]

cb_z_m = fig_m.colorbar(
    plt.cm.ScalarMappable(norm=norm_z, cmap=cmap_z),
    ax=axs_m,
    label=r"$\langle Z_i(t) \rangle$",
    fraction=0.10,
    pad=0.03,
    aspect=15,
)
cb_o_m = fig_m.colorbar(
    plt.cm.ScalarMappable(norm=norm_occ_m, cmap=cmap_occ_m),
    ax=axh_m,
    label=rf"Reduced occupation $P_{N_ACTIVE}(t)$",
    extend="max",
    fraction=0.10,
    pad=0.03,
    aspect=15,
)
for _cb in (cb_z_m, cb_o_m):
    _cb.outline.set_visible(False)
    _cb.outline.set_linewidth(0)

# As render_movie.py: freeze the layout, grow the Hilbert limits so the
# initial-state marker is not clipped, and trim the canvas once.
fig_m.canvas.draw()
fig_m.set_constrained_layout(False)
MARKER_RADIUS_PT = 7.5  # ring marker radius + edge + margin, pt
for _pass in range(2):
    fig_m.canvas.draw()
    renderer = fig_m.canvas.get_renderer()
    for ax_h, XY_h, alpha_h in hilbert_axes_info:
        bbox = ax_h.get_window_extent(renderer)
        xlo, xhi = ax_h.get_xlim()
        ylo, yhi = ax_h.get_ylim()
        px_per_data_x = bbox.width / (xhi - xlo)
        px_per_data_y = bbox.height / (yhi - ylo)
        radius_px = MARKER_RADIUS_PT * fig_m.dpi / 72.0
        need_x = radius_px / px_per_data_x
        need_y = radius_px / px_per_data_y
        ax_x, ay_y = XY_h[alpha_h]
        new_xlo = min(xlo, ax_x - need_x)
        new_xhi = max(xhi, ax_x + need_x)
        new_ylo = min(ylo, ay_y - need_y)
        new_yhi = max(yhi, ay_y + need_y)
        if (new_xlo, new_xhi, new_ylo, new_yhi) != (xlo, xhi, ylo, yhi):
            ax_h.set_xlim(new_xlo, new_xhi)
            ax_h.set_ylim(new_ylo, new_yhi)

fig_m.canvas.draw()
renderer = fig_m.canvas.get_renderer()
_probe = fig_m.text(0.5, 0.5, "Jt = 000.0", ha="center", va="center")
fig_m.canvas.draw()
TIME_TEXT_PAD_PX = _probe.get_window_extent(renderer).height * 1.3
_probe.remove()

TOP_PAD_PX, BOTTOM_PAD_PX, SIDE_PAD_PX = 10, 16, 8
tight_boxes = [ax.get_tightbbox(renderer) for ax in fig_m.axes]
x0 = min(b.x0 for b in tight_boxes) - SIDE_PAD_PX
x1 = max(b.x1 for b in tight_boxes) + SIDE_PAD_PX
y0 = min(b.y0 for b in tight_boxes) - BOTTOM_PAD_PX
y1 = max(b.y1 for b in tight_boxes) + TOP_PAD_PX + TIME_TEXT_PAD_PX
new_W_px, new_H_px = x1 - x0, y1 - y0

for ax in fig_m.axes:
    old_box = ax.get_window_extent(renderer)
    ax.set_position(
        [
            (old_box.x0 - x0) / new_W_px,
            (old_box.y0 - y0) / new_H_px,
            old_box.width / new_W_px,
            old_box.height / new_H_px,
        ]
    )
fig_m.set_size_inches(new_W_px / fig_m.dpi, new_H_px / fig_m.dpi, forward=True)
fig_m.canvas.draw()

time_text_m = fig_m.text(0.5, 0.995, "", ha="center", va="top")

In [ ]:
print(
    f"Rendering {N_FRAMES} frames at {FPS} fps "
    f"(~{N_FRAMES / FPS:.0f} s of video)"
)


def _update(k):
    time_text_m.set_text(f"$Jt = {frame_times[k]:6.1f}$")
    line_m.set_data(frame_times[: k + 1], fid_t[: k + 1])
    mask = np.zeros(len(frame_times), dtype=bool)
    mask[k + 1 :] = True
    z_masked = np.ma.masked_array(z_t, mask=np.broadcast_to(mask, z_t.shape))
    mesh_m.set_array(z_masked.ravel())
    sc_m.set_array(np.maximum(marg_prob[:, k], 1e-6))
    return []


ani_m = FuncAnimation(fig_m, _update, frames=N_FRAMES, blit=False)
movie_path = FIGURES / "movie-03.mp4"
ani_m.save(movie_path, writer=FFMpegWriter(fps=FPS), dpi=300)
print("saved", movie_path, movie_path.stat().st_size / 1e6, "MB")